In [0]:
from pyspark.sql.functions import *


path = "/Volumes/workspace/0_bronze/src_systems/streaming_demo/"
schema = "cust_id string, product string, price double, timestamp timestamp"

# 1. Setup the Read Stream

streaming_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .schema(schema)
  .load(path+"*.json"))

#display(streaming_df)

(streaming_df.writeStream
  .format("delta")                       
  .outputMode("append")
  .option("checkpointLocation", f"{path}/checkpoints/")
  .trigger(availableNow=True)           
  .toTable("workspace.0_bronze.stream_cust_sales_data"))

In [0]:


# Quality rules - drop records that fail any check
rules = {
    "valid_cust_id": "cust_id IS NOT NULL",
    "valid_product": "Len(product) >= 6",
    "valid_price": "price > 0",
    "valid_timestamp": "timestamp > '2026-04-01'"
}

# Read stream from bronze table
bronze_stream = spark.readStream.table("workspace.0_bronze.stream_cust_sales_data")

# Define quality condition (all rules must pass)
quality_condition = (
    col("cust_id").isNotNull() &           # valid_cust_id
    (length(col("product")) >= 6) &        # valid_product
    (col("price") > 0) &                   # valid_price
    (col("timestamp") > "2026-04-01")     # valid_timestamp
)

# Build failure reasons array - collect all failures
stream_with_quality = (bronze_stream
    .withColumn("quality_pass", quality_condition)
    .withColumn("failure_reasons_array", 
        array(
            expr("CASE WHEN cust_id IS NULL THEN 'invalid_cust_id' END"),
            expr("CASE WHEN length(product) < 6 THEN 'invalid_product' END"),
            expr("CASE WHEN price <= 0 THEN 'invalid_price' END"),
            expr("CASE WHEN timestamp <= '2026-04-01' THEN 'invalid_timestamp' END")
        )
    )
    .withColumn("failure_reason", 
        array_join(array_compact(col("failure_reasons_array")), ", ")
    )
    .withColumn("failure_reason", 
        expr("CASE WHEN failure_reason = '' THEN NULL ELSE failure_reason END")
    )
    .withColumn("quarantine_timestamp", current_timestamp())
    .drop("failure_reasons_array")
)

# Function to split stream into clean and quarantine
def process_batch(batch_df, batch_id):
    # Clean records go to silver
    clean_df = batch_df.filter(col("quality_pass") == True).drop("quality_pass", "failure_reason", "quarantine_timestamp")
    if clean_df.count() > 0:
        clean_df.write.format("delta").mode("append").saveAsTable("workspace.1_silver.clean_stream_cust_sales_data")
    
    # Failed records go to quarantine
    quarantine_df = batch_df.filter(col("quality_pass") == False)
    if quarantine_df.count() > 0:
        quarantine_df.write.format("delta").mode("append").saveAsTable("workspace.1_silver.stream_cust_sales_quarantine")

# Write stream using foreachBatch to handle both clean and quarantine


(stream_with_quality.writeStream
  .foreachBatch(process_batch)
  .option("checkpointLocation", f"{path}/checkpoints/silver_with_quarantine/")
  .trigger(availableNow=True)
  .start())

In [0]:
%skip
# 2. Transformation: Group by product and a 1-minute window
windowed_counts = (streaming_df
  .groupBy(
      window(col("timestamp"), "1 minute"),
      col("cust_id")
  )
  .sum("price")
  .select("window.start", "cust_id", col("sum(price)").alias("total_revenue"))
)